# Football matches betting odds data.

## The goal of this project is to take a look at what's happening inside a bookmaker's football odds. Let's investigate English Premier League football league matches throughout many seasons.

## Betting odds dataset:
* [website](https://www.football-data.co.uk/englandm.php)
* [dataset description](https://www.football-data.co.uk/notes.txt)
* [Premier League 2021/2022](https://www.football-data.co.uk/mmz4281/2122/E0.csv)

Downloading the dataset.

In [1]:
import pandas as pd

EPL = pd.read_csv('https://www.football-data.co.uk/mmz4281/2122/E0.csv')

Useful features:
* HomeTeam
* AwayTeam
* B365CH
* B365CD
* B365CA

These are so called closing odds on home team win, draw, away team win respectively, provided by B365 bookmaker.

## Matches statistics dataset:
* [kaggle](https://www.kaggle.com/datasets/josephvm/english-premier-league-game-events-and-results)
* [matches](https://www.kaggle.com/datasets/josephvm/english-premier-league-game-events-and-results?select=matches.csv)

Downloading the dataset.

In [2]:
import pandas as pd

matches = pd.read_csv('matches.csv')

Useful features:
* home
* away
* year
* time (utc)
* attendance
* home_score
* away_score
* home_starting_N $\quad$ (1 $\leqslant$ N $\leqslant$ 11)
* home_bench_N $\quad$ (1 $\leqslant$ N $\leqslant$ 5)
* away_starting_N $\quad$ (1 $\leqslant$ N $\leqslant$ 11)
* away_bench_N $\quad$ (1 $\leqslant$ N $\leqslant$ 5)
* home_formation
* away_formation
* home_goal_scorers
* away_goal_scorers

In [30]:
features = [
    'home',
    'away',
    'year',
    'attendance',
    'home_score',
    'away_score'
]
for i in range(1, 12):
    features.append('home_starting_' + str(i))
for i in range(1, 6):
    features.append('home_bench_' + str(i))
for i in range(1, 12):
    features.append('away_starting_' + str(i))
for i in range(1, 6):
    features.append('away_bench_' + str(i))
features += [
    'home_formation',
    'away_formation',
    'home_goal_scorers',
    'away_goal_scorers'
]

We only consider the last 10 seasons (2021/2022 to 2012/2013) to get approximately 3800 matches overall.

In [4]:
def make_dataset(YYYY):
    dataset = {}
    for key in features:
      dataset[key] = []

    for i in range(len(matches)):
        match_ = matches.iloc[i]['id'].split(',')
        match = [''] * 155
        index = 0
        is_in_string = False
        for info in match_:
            info = str(info)
            match[index] += info
            if len(info) == 0:
                index += 1
                continue
            if info[0] == '"':
                is_in_string = True
            if info[-1] == '"':
                is_in_string = False
            if not is_in_string:
                index += 1

        home = match[1]
        away = match[2]
        year = match[4]
        if year != YYYY:
            continue
        attendance = match[6][1:-1]
        home_score = match[12]
        away_score = match[13]
        mark = 35
        home_goal_scorers = match[mark - 4]
        away_goal_scorers = match[mark - 2]
        home_starting = []
        for j in range(mark, mark + 21, 2):
            home_starting.append(match[j])
        home_bench = []
        for j in range(mark + 22, mark + 35, 3):
            home_bench.append(match[j])
        away_starting = []
        for j in range(mark + 37, mark + 58, 2):
            away_starting.append(match[j])
        away_bench = []
        for j in range(mark + 59, mark + 72, 3):
            away_bench.append(match[j])
        home_formation = match[118]
        away_formation = match[119]

        dataset['home'].append(home)
        dataset['away'].append(away)
        dataset['year'].append(year)
        dataset['attendance'].append(attendance)
        dataset['home_score'].append(home_score)
        dataset['away_score'].append(away_score)
        for j in range(1, 12):
            dataset['home_starting_' + str(j)].append(home_starting[j - 1])
        for j in range(1, 12):
            dataset['away_starting_' + str(j)].append(away_starting[j - 1])
        for j in range(1, 6):
            dataset['home_bench_' + str(j)].append(home_bench[j - 1])
        for j in range(1, 6):
            dataset['away_bench_' + str(j)].append(away_bench[j - 1])
        dataset['home_formation'].append(home_formation)
        dataset['away_formation'].append(away_formation)
        dataset['home_goal_scorers'].append(home_goal_scorers)
        dataset['away_goal_scorers'].append(away_goal_scorers)

    return pd.DataFrame.from_dict(dataset)

In [5]:
dataset_2012 = make_dataset('2012')
dataset_2013 = make_dataset('2013')
dataset_2014 = make_dataset('2014')
dataset_2015 = make_dataset('2015')
dataset_2016 = make_dataset('2016')
dataset_2017 = make_dataset('2017')
dataset_2018 = make_dataset('2018')
dataset_2019 = make_dataset('2019')
dataset_2020 = make_dataset('2020')
dataset_2021 = make_dataset('2021')

Let's add some features, which somehow show each team performance, into our dataset.

First, we want to add features __HomePoints4__ and __AwayPoints4__, which is the number of points home or away team has gained in the last 4 matches.

In [6]:
def add_features_points(dataset):
    HomePoints4 = []
    AwayPoints4 = []

    Points4Dict = {}

    for i in range(len(dataset)):
        match = dataset.iloc[i]
        home, away = match['home'], match['away']
        HG, AG = int(match['home_score']), int(match['away_score'])
        HP, AP = 1, 1
        if HG > AG:
            HP, AP = 3, 0
        if HG < AG:
            HP, AP = 0, 3
        
        if home not in Points4Dict:
            Points4Dict[home] = [0, 0, 0, 0]
        if away not in Points4Dict:
            Points4Dict[away] = [0, 0, 0, 0]
        
        HomePoints4.append(sum(Points4Dict[home]))
        AwayPoints4.append(sum(Points4Dict[away]))
        
        Points4Dict[home] = [HP, Points4Dict[home][0], Points4Dict[home][1], Points4Dict[home][2]]
        Points4Dict[away] = [AP, Points4Dict[away][0], Points4Dict[away][1], Points4Dict[away][2]]

    dataset['HomePoints4'] = HomePoints4
    dataset['AwayPoints4'] = AwayPoints4

    return dataset

In [7]:
dataset_2012 = add_features_points(dataset_2012)
dataset_2013 = add_features_points(dataset_2013)
dataset_2014 = add_features_points(dataset_2014)
dataset_2015 = add_features_points(dataset_2015)
dataset_2016 = add_features_points(dataset_2016)
dataset_2017 = add_features_points(dataset_2017)
dataset_2018 = add_features_points(dataset_2018)
dataset_2019 = add_features_points(dataset_2019)
dataset_2020 = add_features_points(dataset_2020)
dataset_2021 = add_features_points(dataset_2021)

Again, we add new features __HomeCurrentStandings__ and __AwayCurrentStandings__, team place in the table.

In [8]:
def result(HG, AG):
    if HG > AG:
        return (3, 0)
    if HG < AG:
        return (0, 3)
    return (1, 1)

def add_features_standings(dataset):
    teams = []
    for match in range(len(dataset)):
        home = dataset.iloc[match]['home']
        if home not in teams:
            teams.append(home)
    teams.sort(reverse=True)

    HomeCurrentStandings = []
    AwayCurrentStandings = []

    standings = []
    for team in teams:
        standings.append([0, 0, teams.index(team)])

    for match in range(len(dataset)):
        HG, AG = int(dataset.iloc[match]['home_score']), int(dataset.iloc[match]['away_score'])
        points = result(HG, AG)

        standings.sort(reverse=True)
        home, away = dataset.iloc[match]['home'], dataset.iloc[match]['away']

        for i in range(len(standings)):
            if home == teams[standings[i][2]]:
                HomeCurrentStandings.append(standings.index(standings[i]))
                standings[i] = [
                    standings[i][0] + points[0],
                    standings[i][1] + HG - AG,
                    standings[i][2]
                ]
                break
        for i in range(len(standings)):
            if away == teams[standings[i][2]]:
                AwayCurrentStandings.append(standings.index(standings[i]))
                standings[i] = [
                    standings[i][0] + points[1],
                    standings[i][1] + AG - HG,
                    standings[i][2]
                ]
                break

    print('Season final standings:\n')
    for i in range(len(standings)):
        print('#', i + 1, sep='')
        print('team:', teams[standings[i][2]])
        print('points:', standings[i][0])
        print('goal difference:', standings[i][1])
        print()

    dataset['HomeCurrentStandings'] = HomeCurrentStandings
    dataset['AwayCurrentStandings'] = AwayCurrentStandings

    return dataset

In [9]:
dataset_2012 = add_features_standings(dataset_2012)
dataset_2013 = add_features_standings(dataset_2013)
dataset_2014 = add_features_standings(dataset_2014)
dataset_2015 = add_features_standings(dataset_2015)
dataset_2016 = add_features_standings(dataset_2016)
dataset_2017 = add_features_standings(dataset_2017)
dataset_2018 = add_features_standings(dataset_2018)
dataset_2019 = add_features_standings(dataset_2019)
dataset_2020 = add_features_standings(dataset_2020)
dataset_2021 = add_features_standings(dataset_2021)

Season final standings:

#1
team: Manchester United
points: 89
goal difference: 43

#2
team: Manchester City
points: 78
goal difference: 32

#3
team: Chelsea
points: 75
goal difference: 36

#4
team: Arsenal
points: 73
goal difference: 35

#5
team: Tottenham Hotspur
points: 72
goal difference: 20

#6
team: Everton
points: 63
goal difference: 15

#7
team: Liverpool
points: 61
goal difference: 28

#8
team: West Bromwich Albion
points: 49
goal difference: -4

#9
team: Swansea City
points: 46
goal difference: -4

#10
team: West Ham United
points: 46
goal difference: -8

#11
team: Norwich City
points: 44
goal difference: -17

#12
team: Fulham
points: 43
goal difference: -10

#13
team: Stoke City
points: 42
goal difference: -11

#14
team: Southampton
points: 41
goal difference: -11

#15
team: Newcastle United
points: 41
goal difference: -23

#16
team: Aston Villa
points: 41
goal difference: -22

#17
team: Sunderland
points: 39
goal difference: -13

#18
team: Wigan Athletic
points: 36
goal dif

We add two more new features: __HomeLineupGoals__ and __AwayLineupGoals__, which show the number of goals home or away team lineup players have scored totally so far.

In [10]:
def add_features_goals(dataset):
    HomeLineupGoals = []
    AwayLineupGoals = []

    player_goals = {}

    for match in range(len(dataset)):
        HLG, ALG = 0, 0
        for j in range(1, 12):
            player = dataset.iloc[match]['home_starting_' + str(j)]
            HLG += player_goals.get(player, 0)
        for j in range(1, 12):
            player = dataset.iloc[match]['away_starting_' + str(j)]
            ALG += player_goals.get(player, 0)
        HomeLineupGoals.append(HLG)
        AwayLineupGoals.append(ALG)

        home_goal_scorers, away_goal_scorers = dataset.iloc[match]['home_goal_scorers'].split(':'), dataset.iloc[match]['away_goal_scorers'].split(':')
        for player in home_goal_scorers:
            if player == '':
                break
            player_goals[player] = player_goals.get(player, 0) + 1
        for player in away_goal_scorers:
            if player == '':
                break
            player_goals[player] = player_goals.get(player, 0) + 1

    dataset['HomeLineupGoals'] = HomeLineupGoals
    dataset['AwayLineupGoals'] = AwayLineupGoals

    top = []
    for player in player_goals:
        top.append((-player_goals[player], player))
    top.sort()

    print('Top 5 goal-scorers:')
    for i in range(5):
        print('#', i + 1, top[i][1], -top[i][0], 'goals')
    print()

    return dataset

In [11]:
dataset_2012 = add_features_goals(dataset_2012)
dataset_2013 = add_features_goals(dataset_2013)
dataset_2014 = add_features_goals(dataset_2014)
dataset_2015 = add_features_goals(dataset_2015)
dataset_2016 = add_features_goals(dataset_2016)
dataset_2017 = add_features_goals(dataset_2017)
dataset_2018 = add_features_goals(dataset_2018)
dataset_2019 = add_features_goals(dataset_2019)
dataset_2020 = add_features_goals(dataset_2020)
dataset_2021 = add_features_goals(dataset_2021)

Top 5 goal-scorers:
# 1 Robin van Persie 26 goals
# 2 Luis Suárez 23 goals
# 3 Gareth Bale 22 goals
# 4 Christian Benteke 19 goals
# 5 Michu 18 goals

Top 5 goal-scorers:
# 1 Luis Suárez 31 goals
# 2 Daniel Sturridge 21 goals
# 3 Yaya Touré 20 goals
# 4 Wayne Rooney 18 goals
# 5 Sergio Agüero 17 goals

Top 5 goal-scorers:
# 1 Sergio Agüero 25 goals
# 2 Harry Kane 22 goals
# 3 Diego Costa 20 goals
# 4 Charlie Austin 18 goals
# 5 Alexis Sánchez 16 goals

Top 5 goal-scorers:
# 1 Harry Kane 26 goals
# 2 Jamie Vardy 24 goals
# 3 Sergio Agüero 24 goals
# 4 Romelu Lukaku 18 goals
# 5 Olivier Giroud 17 goals

Top 5 goal-scorers:
# 1 Harry Kane 29 goals
# 2 Romelu Lukaku 25 goals
# 3 Alexis Sánchez 24 goals
# 4 Diego Costa 20 goals
# 5 Sergio Agüero 20 goals

Top 5 goal-scorers:
# 1 Mohamed Salah 32 goals
# 2 Harry Kane 30 goals
# 3 Sergio Agüero 21 goals
# 4 Jamie Vardy 20 goals
# 5 Raheem Sterling 18 goals

Top 5 goal-scorers:
# 1 Mohamed Salah 22 goals
# 2 Pierre-Emerick Aubameyang 22 goals


Finally we want to consider the rating of each team match opponent. Suppose the opponent's current table standing is $1 \leqslant n \leqslant 20$. Then we calculate a goal against as $1.5 - \frac{n - 0.5}{20}$ and a goal conceded as $0.5 + \frac{n - 0.5}{20}$. Example:
* $n = 1$. Goal against is equal to $1.475$, goal conceded is equal to $0.525$.
* $n = 10$. Goal against is equal to $1.025$, goal conceded is equal to $0.975$.
* $n = 20$. Goal against is equal to $0.525$, goal conceded is equal to $1.475$.

Adding the features __HomePerformance4__ and __AwayPerformance4__, which show home team and away team goal difference in the last 4 matches, considering the opponent's rating.

In [12]:
def against(n):
    return 1.5 - (n - 0.5) / 20

def conceded(n):
    return 0.5 + (n - 0.5) / 20

def add_features_performance(dataset):
    HomePerformance4 = []
    AwayPerformance4 = []

    Performance4Dict = {}

    for i in range(len(dataset)):
        match = dataset.iloc[i]
        home, away = match['home'], match['away']
        HG, AG = int(match['home_score']), int(match['away_score'])
        home_standing, away_standing = match['HomeCurrentStandings'], match['AwayCurrentStandings']

        HP = HG * against(away_standing) - AG * conceded(away_standing)
        AP = AG * against(home_standing) - HG * conceded(home_standing)
        
        if home not in Performance4Dict:
            Performance4Dict[home] = [0, 0, 0, 0]
        if away not in Performance4Dict:
            Performance4Dict[away] = [0, 0, 0, 0]
        
        HomePerformance4.append(sum(Performance4Dict[home]))
        AwayPerformance4.append(sum(Performance4Dict[away]))
        
        Performance4Dict[home] = [HP, Performance4Dict[home][0], Performance4Dict[home][1], Performance4Dict[home][2]]
        Performance4Dict[away] = [AP, Performance4Dict[away][0], Performance4Dict[away][1], Performance4Dict[away][2]]

    dataset['HomePerformance4'] = HomePerformance4
    dataset['AwayPerformance4'] = AwayPerformance4

    return dataset

In [13]:
dataset_2012 = add_features_performance(dataset_2012)
dataset_2013 = add_features_performance(dataset_2013)
dataset_2014 = add_features_performance(dataset_2014)
dataset_2015 = add_features_performance(dataset_2015)
dataset_2016 = add_features_performance(dataset_2016)
dataset_2017 = add_features_performance(dataset_2017)
dataset_2018 = add_features_performance(dataset_2018)
dataset_2019 = add_features_performance(dataset_2019)
dataset_2020 = add_features_performance(dataset_2020)
dataset_2021 = add_features_performance(dataset_2021)

Bringing the dataset into one.

In [14]:
dataset = pd.concat([
    dataset_2012,
    dataset_2013,
    dataset_2014,
    dataset_2015,
    dataset_2016,
    dataset_2017,
    dataset_2018,
    dataset_2019,
    dataset_2020,
    dataset_2021
], axis=0)

Suppose there are $N$ distinct players. Let's create $N$ new features indicating whether a certain player is present in the home team lineup. Similarly we'll create $N$ more new features for away teams.

In [15]:
players_N = set()

for match in range(len(dataset)):
    for j in range(1, 12):
        player = dataset.iloc[match]['home_starting_' + str(j)]
        players_N.add(player)
    for j in range(1, 12):
        player = dataset.iloc[match]['away_starting_' + str(j)]
        players_N.add(player)

print(len(players_N))

1646


In [16]:
players_home = {}
for player in players_N:
    players_home[player + ' home'] = [0] * len(dataset)
players_away = {}
for player in players_N:
    players_away[player + ' away'] = [0] * len(dataset)

for match in range(len(dataset)):
    for j in range(1, 12):
        player = dataset.iloc[match]['home_starting_' + str(j)]
        players_home[player + ' home'][match] = 1
    for j in range(1, 12):
        player = dataset.iloc[match]['away_starting_' + str(j)]
        players_away[player + ' away'][match] = 1

for name in players_home:
    dataset[name] = players_home[name]
for name in players_away:
    dataset[name] = players_away[name]

<ipython-input-16-613636d6eec0>:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[name] = players_home[name]
<ipython-input-16-613636d6eec0>:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[name] = players_away[name]


Preparing for CatBoostClassifier.

In [17]:
!pip install catboost

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 MB 11.1 MB/s eta 0:00:00


In [31]:
import numpy as np

features.remove('attendance')
features.remove('home_score')
features.remove('away_score')
for i in range(1, 12):
    features.remove('home_starting_' + str(i))
for i in range(1, 12):
    features.remove('away_starting_' + str(i))
for i in range(1, 6):
    features.remove('home_bench_' + str(i))
for i in range(1, 6):
    features.remove('away_bench_' + str(i))
features.remove('home_goal_scorers')
features.remove('away_goal_scorers')

features.append('HomePoints4')
features.append('AwayPoints4')
features.append('HomeCurrentStandings')
features.append('AwayCurrentStandings')
features.append('HomeLineupGoals')
features.append('AwayLineupGoals')
features.append('HomePerformance4')
features.append('AwayPerformance4')

for player in players_N:
    features.append(player + ' home')
    features.append(player + ' away')

In [32]:
X = dataset[features]
X = X.to_numpy()

In [33]:
def decode_result(HG, AG):
    if HG > AG:
        return 0
    if HG == AG:
        return 1
    if HG < AG:
        return 2
    return -1

y = []
for HG, AG in dataset[['home_score', 'away_score']].to_numpy():
    y.append([decode_result(int(HG), int(AG))])
y = np.asarray(y)

In [34]:
for i in range(len(features)):
    print(i, features[i])

0 home
1 away
2 year
3 home_formation
4 away_formation
5 HomePoints4
6 AwayPoints4
7 HomeCurrentStandings
8 AwayCurrentStandings
9 HomeLineupGoals
10 AwayLineupGoals
11 HomePerformance4
12 AwayPerformance4
13  home
14  away
15 Daniel Potts home
16 Daniel Potts away
17 James Collins home
18 James Collins away
19 Clint Hill home
20 Clint Hill away
21 Mobido Diakité home
22 Mobido Diakité away
23 Danny Guthrie home
24 Danny Guthrie away
25 Grant Holt home
26 Grant Holt away
27 Martin Dúbravka home
28 Martin Dúbravka away
29 Cheikhou Kouyaté home
30 Cheikhou Kouyaté away
31 Magnus Wolff Eikrem home
32 Magnus Wolff Eikrem away
33 Sam Johnstone home
34 Sam Johnstone away
35 Diogo Dalot home
36 Diogo Dalot away
37 Mehdi Abeid home
38 Mehdi Abeid away
39 Michael Essien home
40 Michael Essien away
41 Shandon Baptiste home
42 Shandon Baptiste away
43 David Meyler home
44 David Meyler away
45 Willian José home
46 Willian José away
47 Kyle Walker-Peters home
48 Kyle Walker-Peters away
49 Peter Ode

Removing empty names.

In [35]:
features.remove(' home')
features.remove(' away')

In [36]:
print(len(X))

3797


We will train the model on the first 380 $\cdot$ 9 + 10 $\cdot$ 4 = 3460 matches (first 9 seasons + first 4 matchdays in season 2021/2022), bet on the last 3797 - 3460 = 337 matches.

In [37]:
N = 3460

In [38]:
X_train, X_test = X[:N], X[N:]
y_train, y_test = y[:N], y[N:]

Training the model.

In [39]:
from catboost import CatBoostClassifier, Pool

train_data = X_train
test_data = X_test

cat_features = [
    0, #home
    1, #away
    2, #year
    3, #home_formation
    4, #away_formation
    5, #HomePoints4
    6, #AwayPoints4
    7, #HomeCurrentStandings
    8  #AwayCurrentStandings
] + list(range(13, len(features))) #players

train_label = y_train
test_label = y_test

train_dataset = Pool(
    data=train_data,
    label=train_label,
    cat_features=cat_features
)
test_dataset = Pool(
    data=test_data,
    label=test_label,
    cat_features=cat_features
)

model = CatBoostClassifier(
    iterations=100,
    learning_rate=1,
    depth=8,
    loss_function='MultiClass',
    random_seed=42
)

model.fit(train_dataset)

preds_class = model.predict(test_dataset)
preds_proba = model.predict_proba(test_dataset)

0:	learn: 1.0289166	total: 106ms	remaining: 10.5s
1:	learn: 1.0125606	total: 199ms	remaining: 9.76s
2:	learn: 0.9895950	total: 291ms	remaining: 9.4s
3:	learn: 0.9848225	total: 388ms	remaining: 9.32s
4:	learn: 0.9662448	total: 483ms	remaining: 9.18s
5:	learn: 0.9499682	total: 573ms	remaining: 8.98s
6:	learn: 0.9436351	total: 670ms	remaining: 8.91s
7:	learn: 0.9315161	total: 759ms	remaining: 8.73s
8:	learn: 0.9273284	total: 855ms	remaining: 8.65s
9:	learn: 0.9182976	total: 958ms	remaining: 8.62s
10:	learn: 0.9070955	total: 1.08s	remaining: 8.71s
11:	learn: 0.8985407	total: 1.17s	remaining: 8.6s
12:	learn: 0.8883024	total: 1.26s	remaining: 8.44s
13:	learn: 0.8805718	total: 1.35s	remaining: 8.3s
14:	learn: 0.8741711	total: 1.46s	remaining: 8.26s
15:	learn: 0.8678053	total: 1.55s	remaining: 8.14s
16:	learn: 0.8590691	total: 1.65s	remaining: 8.04s
17:	learn: 0.8532482	total: 1.74s	remaining: 7.94s
18:	learn: 0.8471485	total: 1.84s	remaining: 7.85s
19:	learn: 0.8360581	total: 1.95s	remaining:

How many results has the model guessed right?

In [40]:
guessed = 0

for match in range(len(test_data)):
    result = y_test[match]
    preds = np.argmax(preds_proba[match])

    if result == preds:
        guessed += 1

print(guessed, 'out of', len(test_data), 'or', guessed / len(test_data))

161 out of 337 or 0.47774480712166173


Getting the probabilities of results.

In [41]:
probs = {}

for match in range(len(test_data)):
    home, away = test_data[match][0], test_data[match][1]
    H_prob, D_prob, A_prob = preds_proba[match]
    probs[(home, away)] = (H_prob, D_prob, A_prob)

Notice that teams in EPL dataset and "matches" dataset can have different names. So let's fix that.

In [42]:
teams_EPL = []
for match in range(len(EPL)):
    home = EPL.iloc[match]['HomeTeam']
    if home not in teams_EPL:
        teams_EPL.append(home)
teams_EPL.sort()
print(teams_EPL)

teams_matches = []
for match in range(N, len(dataset)):
    home = dataset.iloc[match]['home']
    if home not in teams_matches:
        teams_matches.append(home)
teams_matches.sort()
print(teams_matches)

['Arsenal', 'Aston Villa', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Leeds', 'Leicester', 'Liverpool', 'Man City', 'Man United', 'Newcastle', 'Norwich', 'Southampton', 'Tottenham', 'Watford', 'West Ham', 'Wolves']
['Arsenal', 'Aston Villa', 'Brentford', 'Brighton & Hove Albion', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Leeds United', 'Leicester City', 'Liverpool', 'Manchester City', 'Manchester United', 'Newcastle United', 'Norwich City', 'Southampton', 'Tottenham Hotspur', 'Watford', 'West Ham United', 'Wolverhampton Wanderers']


In [43]:
team = {}
for i in range(20):
    team[teams_matches[i]] = teams_EPL[i]

Calculating the [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) error between CatBoostClassifier and B365 probabilities.

In [44]:
MAE = 0

for match in range(N, len(dataset)):
    home, away = dataset.iloc[match]['home'], dataset.iloc[match]['away']
    if (home, away) not in probs:
        continue

    H_prob, D_prob, A_prob = probs[(home, away)]

    index = 0
    for i in range(len(EPL)):
        if (team[home], team[away]) == (EPL.iloc[i]['HomeTeam'], EPL.iloc[i]['AwayTeam']):
            index = i
            break
    H_coef, D_coef, A_coef = EPL.iloc[index]['B365CH'], EPL.iloc[index]['B365CD'], EPL.iloc[index]['B365CA']

    MAE += abs(H_prob - 1 / H_coef)
    MAE += abs(D_prob - 1 / D_coef)
    MAE += abs(A_prob - 1 / A_coef)

MAE /= (len(dataset) - N) * 3

print(int(MAE * 100), '%', sep='')

14%


Calculating our win using Kelly criterion and the following algorithm.

In [45]:
def Kelly(prob, coef):
    return (coef * prob - 1) / (coef - 1)

In [46]:
def algo(prob, coef):
    return 1 / prob < coef

Obviously most people bet on either home team win or away team win. So draw result is usually underrated. We'll bet on draws only.

In [47]:
WIN = 0
bank = 0

for match in range(N, len(dataset)):
    home, away = dataset.iloc[match]['home'], dataset.iloc[match]['away']
    if (home, away) not in probs:
        continue

    H_prob, D_prob, A_prob = probs[(home, away)]

    index = 0
    for i in range(len(EPL)):
        if (team[home], team[away]) == (EPL.iloc[i]['HomeTeam'], EPL.iloc[i]['AwayTeam']):
            index = i
            break
    H_coef, D_coef, A_coef = EPL.iloc[index]['B365CH'], EPL.iloc[index]['B365CD'], EPL.iloc[index]['B365CA']

    result = ''
    HG, AG = int(dataset.iloc[match]['home_score']), int(dataset.iloc[match]['away_score'])
    if HG > AG:
        result = 'H'
    if HG == AG:
        result = 'D'
    if HG < AG:
        result = 'A'

    bet = [Kelly(H_prob, H_coef), Kelly(D_prob, D_coef), Kelly(A_prob, A_coef)]

    win = 0
    if algo(D_prob, D_coef):
        bank += bet[1]
        win -= bet[1]
        if result == 'D':
            win += D_coef * bet[1]
    WIN += win

print(WIN, ' from ', bank, ' or +', WIN / bank * 100, '%', sep='')

5.989750692050718 from 24.68712303662847 or +24.26265176044888%


Let's manually investigate the results of the last 337 matches, which we bet on and lose.

In [48]:
n_lost = 0
n = 0

for match in range(N, len(dataset)):
    home, away = dataset.iloc[match]['home'], dataset.iloc[match]['away']
    if (home, away) not in probs:
        continue

    H_prob, D_prob, A_prob = probs[(home, away)]

    index = 0
    for i in range(len(EPL)):
        if (team[home], team[away]) == (EPL.iloc[i]['HomeTeam'], EPL.iloc[i]['AwayTeam']):
            index = i
            break
    H_coef, D_coef, A_coef = EPL.iloc[index]['B365CH'], EPL.iloc[index]['B365CD'], EPL.iloc[index]['B365CA']

    result = ''
    HG, AG = int(dataset.iloc[match]['home_score']), int(dataset.iloc[match]['away_score'])
    if HG > AG:
        result = 'H'
    if HG == AG:
        result = 'D'
    if HG < AG:
        result = 'A'

    bet = [Kelly(H_prob, H_coef), Kelly(D_prob, D_coef), Kelly(A_prob, A_coef)]

    win = 0
    if algo(D_prob, D_coef):
        bank += bet[1]
        win -= bet[1]
        if result == 'D':
            win += D_coef * bet[1]

    if win < 0:
        n_lost += 1
    if win != 0:
        n += 1

print(n_lost, ' out of ', n, ' or ', n_lost / n * 100, '%', sep='')

91 out of 128 or 71.09375%


You can notice that we train the model only once and then bet. Now let's do the following way:
1. Train the model.
2. For $i$ in {5, 6, ..., 37, 38}:
  1. Bet on matchday $i$ (10 matches).
  2. Calculate the winning from the matchday.
  3. Add these 10 matches to the train data.
  4. Train the model.

Let's do that.

In [49]:
def calculate_winning(matchday):
    N = 380 * 9 + 10 * (matchday - 1)

    X_train, X_test = X[:N], X[N:N + 10]
    y_train, y_test = y[:N], y[N:N + 10]

    train_data = X_train
    test_data = X_test

    cat_features = [
        0, #home
        1, #away
        2, #year
        3, #home_formation
        4, #away_formation
        5, #HomePoints4
        6, #AwayPoints4
        7, #HomeCurrentStandings
        8  #AwayCurrentStandings
    ] + list(range(13, len(features))) #players

    train_label = y_train
    test_label = y_test

    train_dataset = Pool(
        data=train_data,
        label=train_label,
        cat_features=cat_features
    )
    test_dataset = Pool(
        data=test_data,
        label=test_label,
        cat_features=cat_features
    )

    model = CatBoostClassifier(
        iterations=100,
        learning_rate=1,
        depth=8,
        loss_function='MultiClass',
        random_seed=42
    )

    print('Matchday', matchday)
    model.fit(train_dataset)
    print()

    preds_class = model.predict(test_dataset)
    preds_proba = model.predict_proba(test_dataset)

    probs = {}

    for match in range(len(test_data)):
        home, away = test_data[match][0], test_data[match][1]
        H_prob, D_prob, A_prob = preds_proba[match]
        probs[(home, away)] = (H_prob, D_prob, A_prob)

    WIN = 0
    bank = 0

    for match in range(N, len(dataset)):
        home, away = dataset.iloc[match]['home'], dataset.iloc[match]['away']
        if (home, away) not in probs:
            continue

        H_prob, D_prob, A_prob = probs[(home, away)]

        index = 0
        for i in range(len(EPL)):
            if (team[home], team[away]) == (EPL.iloc[i]['HomeTeam'], EPL.iloc[i]['AwayTeam']):
                index = i
                break
        H_coef, D_coef, A_coef = EPL.iloc[index]['B365CH'], EPL.iloc[index]['B365CD'], EPL.iloc[index]['B365CA']

        result = ''
        HG, AG = int(dataset.iloc[match]['home_score']), int(dataset.iloc[match]['away_score'])
        if HG > AG:
            result = 'H'
        if HG == AG:
            result = 'D'
        if HG < AG:
            result = 'A'

        bet = [Kelly(H_prob, H_coef), Kelly(D_prob, D_coef), Kelly(A_prob, A_coef)]

        win = 0
        if algo(D_prob, D_coef):
            bank += bet[1]
            win -= bet[1]
            if result == 'D':
                win += D_coef * bet[1]
        WIN += win

    return WIN, bank

WIN = 0
bank = 0
for i in range(5, 39):
    t = calculate_winning(i)
    WIN += t[0]
    bank += t[1]

print(WIN, ' from ', bank, ' or +', WIN / bank * 100, '%', sep='')

Matchday 5
0:	learn: 1.0289166	total: 91.6ms	remaining: 9.07s
1:	learn: 1.0125606	total: 185ms	remaining: 9.06s
2:	learn: 0.9895950	total: 276ms	remaining: 8.91s
3:	learn: 0.9848225	total: 370ms	remaining: 8.89s
4:	learn: 0.9662448	total: 470ms	remaining: 8.93s
5:	learn: 0.9499682	total: 568ms	remaining: 8.89s
6:	learn: 0.9436351	total: 666ms	remaining: 8.85s
7:	learn: 0.9315161	total: 760ms	remaining: 8.74s
8:	learn: 0.9273284	total: 851ms	remaining: 8.6s
9:	learn: 0.9182976	total: 943ms	remaining: 8.48s
10:	learn: 0.9070955	total: 1.03s	remaining: 8.37s
11:	learn: 0.8985407	total: 1.13s	remaining: 8.3s
12:	learn: 0.8883024	total: 1.24s	remaining: 8.29s
13:	learn: 0.8805718	total: 1.33s	remaining: 8.17s
14:	learn: 0.8741711	total: 1.42s	remaining: 8.05s
15:	learn: 0.8678053	total: 1.53s	remaining: 8.02s
16:	learn: 0.8590691	total: 1.62s	remaining: 7.9s
17:	learn: 0.8532482	total: 1.71s	remaining: 7.8s
18:	learn: 0.8471485	total: 1.81s	remaining: 7.71s
19:	learn: 0.8360581	total: 1.9s	